In [1]:
# Install dependencies if needed
# !pip install langchain langchain-experimental langchain-chroma pillow open_clip_torch torch matplotlib unstructured pydantic
import os
from textbook_loading import (
    load_book,
    clean_and_categorize_elements,
    summarize_elements,
    store_in_chromadb,
    delete_irrelevant_images,
)

In [2]:
pdf_file = './data/shortExample2_Nutrition_20Pgs.pdf'
image_output_dir = './figures/Nutrition_fastUnstruct'
chroma_persist_dir = './chroma/Nutrition_fastUnstruct/'

# Make sure the data directory exists
assert os.path.exists('./data'), "Error: './data' directory not found."
assert os.path.exists(pdf_file), f"Error: PDF file not found at {pdf_file}."

In [3]:
print("📝 Unstructuring textbooks, filtering junks, semanic chunking...")
raw_pdf_elements = load_book(pdf_file, image_output_dir)
print("🎉 1.process_pdf_with_semantic_chunking complete.")


📝 Unstructuring textbooks, filtering junks, semanic chunking...


The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


🎉 1.process_pdf_with_semantic_chunking complete.


In [4]:
# Clean and categorize
texts, tables, images_raw = clean_and_categorize_elements(raw_pdf_elements, window_size=2, min_meaningful_text_length=75)

In [5]:
# Summarize, store, etc.
text_summaries, table_summaries, image_paths, relevant_images_to_summarize, image_summaries = summarize_elements(
    texts, tables, images_raw
)

/Users/mas/Desktop/LLM_Veterinary_AI/textbook_loading.py:257: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Texts and Tables Summary Done!
Checking image relevance with local textual context...
Skipping decorative image: ./figures/Nutrition_fastUnstruct/figure-1-1.jpg
Skipping decorative image: ./figures/Nutrition_fastUnstruct/figure-4-2.jpg
Skipping decorative image: ./figures/Nutrition_fastUnstruct/figure-5-3.jpg
Skipping decorative image: ./figures/Nutrition_fastUnstruct/figure-6-4.jpg
Skipping decorative image: ./figures/Nutrition_fastUnstruct/figure-7-5.jpg
Skipping decorative image: ./figures/Nutrition_fastUnstruct/figure-8-6.jpg
Skipping decorative image: ./figures/Nutrition_fastUnstruct/figure-9-7.jpg
Skipping decorative image: ./figures/Nutrition_fastUnstruct/figure-10-8.jpg
Skipping decorative image: ./figures/Nutrition_fastUnstruct/figure-11-9.jpg
Skipping decorative image: ./figures/Nutrition_fastUnstruct/figure-12-10.jpg
Skipping decorative image: ./figures/Nutrition_fastUnstruct/figure-13-11.jpg
Skipping decorative image: ./figures/Nutrition_fastUnstruct/figure-14-13.jpg
Skippi

In [6]:
retriever = store_in_chromadb(
    text_summaries, texts, table_summaries, tables, image_paths,
    relevant_images_to_summarize, image_summaries,
    persist_directory=chroma_persist_dir
)

In [7]:
delete_irrelevant_images(images_raw, relevant_images_to_summarize)

Successfully deleted irrelevant image: ./figures/Nutrition_fastUnstruct/figure-1-1.jpg
Successfully deleted irrelevant image: ./figures/Nutrition_fastUnstruct/figure-4-2.jpg
Successfully deleted irrelevant image: ./figures/Nutrition_fastUnstruct/figure-5-3.jpg
Successfully deleted irrelevant image: ./figures/Nutrition_fastUnstruct/figure-6-4.jpg
Successfully deleted irrelevant image: ./figures/Nutrition_fastUnstruct/figure-7-5.jpg
Successfully deleted irrelevant image: ./figures/Nutrition_fastUnstruct/figure-8-6.jpg
Successfully deleted irrelevant image: ./figures/Nutrition_fastUnstruct/figure-9-7.jpg
Successfully deleted irrelevant image: ./figures/Nutrition_fastUnstruct/figure-10-8.jpg
Successfully deleted irrelevant image: ./figures/Nutrition_fastUnstruct/figure-11-9.jpg
Successfully deleted irrelevant image: ./figures/Nutrition_fastUnstruct/figure-12-10.jpg
Successfully deleted irrelevant image: ./figures/Nutrition_fastUnstruct/figure-13-11.jpg
Successfully deleted irrelevant image

In [11]:
# System sound, when done
sound_file = "/System/Library/Sounds/Glass.aiff"
os.system(f"afplay '{sound_file}'")

0

# Inspecting Retrieved Docs

In [9]:
query = "show me a handsome cat?"
results = retriever.retrieve_multi_modal(query, k=5)

In [10]:
from IPython.display import display, HTML
import os

# 1. Display all images together as thumbnails
image_paths = set()
for res in results:
    if res["modality"] == "image" and os.path.exists(res["summary"]):
        image_paths.add(res["summary"])
    elif res["modality"] == "image_summary":
        img_path = res["original_metadata"].get("image_path")
        if img_path and os.path.exists(img_path):
            image_paths.add(img_path)

if image_paths:
    html_imgs = " ".join(
        f'<img src="{img}" width="100" style="margin:2px; border:1px solid #ccc;">' for img in image_paths
    )
    display(HTML(html_imgs))
else:
    print("No images found in results.")

# 2. Display original text for each text result
print('-'*40, "Retrieved Text Chunks (first 300 chars)", '-'*40)
for res in results:
    if res["modality"] == "text":
        doc_id = res["original_metadata"].get("doc_id")
        original_text = None
        if doc_id and hasattr(retriever, "docstore"):
            doc = retriever.docstore._collection.get(ids=[doc_id], include=["documents"])
            if doc and doc.get("documents") and doc["documents"][0]:
                original_text = doc["documents"][0]
        if not original_text:
            original_text = res["summary"]
        text_display = original_text[:300] + ("..." if len(original_text) > 300 else "")
        print(text_display)
        print('-'*20)

---------------------------------------- Retrieved Text Chunks (first 300 chars) ----------------------------------------
Generic and Private-Label Brands Generic cat foods do not have a brand name. Private-label pet foods carry the names of the stores in which they are sold. These foods provide a list of ingredients as required by law, but most cannot make claims that the food is nutritionally balanced or complete. Ge...
--------------------
VITAMINS AND MINERALS 496 • CAT OWNER’S HOME VETERINARY HANDBOOK D in the skin through exposure to sunlight). Neither of these vitamins should be supplemented without veterinary consultation as overdosing is common and can be toxic. Calcium deficiency is the most frequent nutritional disorder in cat...
--------------------
TYPES OF CAT FOOD There are three types of cat food: dry, semimoist, and canned. To make mean- ingful comparisons, all must be compared on a dry matter basis (see page 501). When the water content is factored out and the product